# SymPy で学ぶ経済数学 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
Python の記号計算ライブラリ **SymPy** を使って、経済学で使う数学（方程式・微分・最適化・積分・行列）を
「手計算の代わりに Python にやらせる」ことで学ぶチュートリアルです。

## 対象者
- 経済数学（微分・積分・行列）を学んでいる、または学び直したい方
- Python の基本（変数、リスト、関数）を理解している方（`python/python_beginner_tutorial.ipynb` 修了程度）
- SymPy を初めて使う方

## このチュートリアルで学ぶこと
0. 環境準備（JupyterLite 用）
1. SymPy とは：記号と式
2. 方程式を解く：市場均衡
3. 微分：限界概念と弾力性
4. 偏微分：生産関数と限界生産力
5. 最適化：一階条件・二階条件・ラグランジュ乗数法
6. 積分：消費者余剰と生産者余剰
7. 極限と級数：現在価値と永久年金
8. 行列：連立方程式と投入産出分析
9. 数値化とグラフ：evalf・lambdify・matplotlib
10. まとめと総合演習

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- 各章の最後に **練習問題** があります。「解答欄」に自分でコードを書いてから、「解答例」を開いて確認しましょう。
- 数式の答えが表示されたら、手計算の結果と見比べてみてください。

---
## 0. 環境準備（JupyterLite 用）

SymPy・NumPy・matplotlib は JupyterLite に同梱されています。日本語のグラフを描くために
`japanize-matplotlib-jlite` もインストールします。

In [ ]:
# JupyterLite 用のパッケージインストール
try:
    import piplite
    await piplite.install(["sympy", "numpy", "matplotlib", "japanize-matplotlib-jlite"])
except ImportError:
    pass

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語表示用（plt の後に import する）
from IPython.display import display

sp.init_printing()  # 数式をきれいに（LaTeX で）表示する設定

print(f"SymPy バージョン: {sp.__version__}")
print(f"NumPy バージョン: {np.__version__}")

---
## 1. SymPy とは：記号と式

**SymPy** は、数値ではなく **記号（文字）のまま** 計算できるライブラリです。
電卓が `2 * 3 = 6` を計算するのに対し、SymPy は `x * x = x**2` や `d/dx x**2 = 2x` のように、
文字を含む式を **代数的に** 扱えます。経済学の教科書に出てくる式の変形・微分・方程式の解法を、
そのまま Python で再現できます。

### 1.1 記号（シンボル）を定義する

計算に使う文字は、最初に `sp.symbols()` で **記号** として宣言します。
`positive=True` のように性質を指定しておくと、SymPy が場合分けをせずに簡潔な答えを出してくれます
（価格や数量は正の値なので、経済学ではこの指定が便利です）。

In [ ]:
x, y = sp.symbols("x y")                      # 一般的な記号
P, Q = sp.symbols("P Q", positive=True)       # 価格と数量（正の値）

expr = 3 * x**2 + 2 * x + 1
print(type(expr))
expr        # セルの最後に式を書くと、きれいに表示される

### 1.2 式の整理：展開・因数分解・簡約・代入

| 関数 | 働き |
|---|---|
| `sp.expand(式)` | 展開する |
| `sp.factor(式)` | 因数分解する |
| `sp.simplify(式)` | できるだけ簡単な形にする |
| `式.subs(記号, 値)` | 記号に値（または別の式）を代入する |

In [ ]:
expr = (x + 2) * (x - 3)
expanded = sp.expand(expr)
print("展開:", expanded)
print("因数分解:", sp.factor(expanded))
print("簡約:", sp.simplify((x**2 - 1) / (x - 1)))

In [ ]:
# 代入：x = 2 のときの値
print(expanded.subs(x, 2))

# 複数の記号に同時に代入（辞書で渡す）
total = 3 * x + 2 * y
print(total.subs({x: 10, y: 5}))

# 記号に「別の式」を代入することもできる
print(total.subs(y, x**2))

### 1.3 数式をきれいに表示する

`sp.init_printing()` を実行済みなので、セルの最後に式を書くか `display()` に渡すと、
教科書のような数式で表示されます。`sp.latex()` を使うと LaTeX のコードも取り出せます
（レポートや論文にそのまま貼り付けられます）。

In [ ]:
U = x**sp.Rational(1, 2) * y**sp.Rational(1, 2)   # コブ・ダグラス型効用関数（1/2 は Rational で正確に）
display(U)
print("LaTeX:", sp.latex(U))

# 注意: 1/2 と書くと Python の小数 0.5 になる。分数のままにしたいときは sp.Rational(1, 2)
print(x**(1 / 2))
print(x**sp.Rational(1, 2))

### 1.4 数学関数と定数

指数関数・対数関数・平方根などは、Python 標準の `math` ではなく **SymPy 版**（`sp.exp`, `sp.log`, `sp.sqrt`）を使います。
円周率やネイピア数も `sp.pi`, `sp.E` として記号のまま扱えます。

In [ ]:
print(sp.exp(x) * sp.exp(2 * x))          # 指数法則が自動で適用される
print(sp.simplify(sp.log(x**2) - sp.log(x)))
print(sp.sqrt(8))                          # 2*sqrt(2) と正確に表す
print(sp.pi, sp.E, sp.oo)                  # 円周率, ネイピア数, 無限大
print(sp.exp(sp.log(5)))                   # 逆関数の関係も認識する

### 練習問題 1

1. 記号 `a`, `b` を定義し、`(a + b)**3` を展開してください。
2. `x**2 + 5*x + 6` を因数分解してください。
3. 総費用関数 `TC = 100 + 10*Q + 0.5*Q**2` を定義し、`Q = 20` のときの総費用を求めてください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
a, b = sp.symbols("a b")
print(sp.expand((a + b)**3))

# 2
print(sp.factor(x**2 + 5 * x + 6))

# 3
TC = 100 + 10 * Q + sp.Rational(1, 2) * Q**2
print(TC.subs(Q, 20))
```

</details>

---
## 2. 方程式を解く：市場均衡

### 2.1 sp.solve と sp.Eq

方程式は `sp.Eq(左辺, 右辺)` で表し、`sp.solve(方程式, 未知数)` で解きます。
`sp.solve(式, 未知数)` と書くと「式 = 0」を解きます。

ある財の需要と供給が次のように与えられているとします。

- 需要関数: $Q_d = 100 - 2P$
- 供給関数: $Q_s = 10 + P$

均衡価格は $Q_d = Q_s$ を満たす $P$ です。

In [ ]:
Qd = 100 - 2 * P      # 需要関数
Qs = 10 + P           # 供給関数

equilibrium = sp.Eq(Qd, Qs)
display(equilibrium)

P_star = sp.solve(equilibrium, P)
print("均衡価格:", P_star)
print("均衡数量:", Qd.subs(P, P_star[0]))

`sp.solve()` の結果は **リスト** で返ります（解が複数あることがあるため）。`[0]` で最初の解を取り出します。

### 2.2 連立方程式

複数の方程式をリストで渡し、未知数もリストで渡します。`dict=True` を付けると、解が辞書で返るので扱いやすくなります。

In [ ]:
# 需要曲線と供給曲線を「P と Q の関係式」として連立で解く
eq_demand = sp.Eq(Q, 100 - 2 * P)
eq_supply = sp.Eq(Q, 10 + P)

sol = sp.solve([eq_demand, eq_supply], [P, Q], dict=True)
print(sol)
print("均衡価格 P* =", sol[0][P], ", 均衡数量 Q* =", sol[0][Q])

### 2.3 パラメータのまま解く

SymPy の強みは、数値を入れずに **文字（パラメータ）のまま** 解けることです。
需要 $Q_d = a - bP$、供給 $Q_s = c + dP$ の均衡価格を一般的に求めてみましょう。

In [ ]:
a, b, c, d = sp.symbols("a b c d", positive=True)

P_general = sp.solve(sp.Eq(a - b * P, c + d * P), P)[0]
display(P_general)

# 具体的な値を代入して確認（a=100, b=2, c=10, d=1）
print(P_general.subs({a: 100, b: 2, c: 10, d: 1}))

### 2.4 グラフで確認する

SymPy の式は `sp.lambdify()` で NumPy 用の関数に変換できます（詳しくは第 9 章）。
需要曲線と供給曲線を描き、交点が均衡になっていることを確かめましょう。

In [ ]:
demand_fn = sp.lambdify(P, Qd, "numpy")
supply_fn = sp.lambdify(P, Qs, "numpy")

prices = np.linspace(0, 50, 100)
plt.figure(figsize=(7, 4.5))
plt.plot(demand_fn(prices), prices, label="需要曲線 Qd = 100 - 2P")
plt.plot(supply_fn(prices), prices, label="供給曲線 Qs = 10 + P")
plt.scatter([40], [30], color="red", zorder=5, label="均衡点 (Q*=40, P*=30)")
plt.xlabel("数量 Q")
plt.ylabel("価格 P")
plt.title("市場均衡")
plt.legend()
plt.grid(True)
plt.show()

### 練習問題 2

1. 需要関数 $Q_d = 200 - 4P$、供給関数 $Q_s = 20 + 2P$ の均衡価格と均衡数量を求めてください。
2. 上の市場で、政府が 1 単位あたり $t = 6$ の従量税を **供給者** に課すと、供給関数は $Q_s = 20 + 2(P - 6)$ になります。新しい均衡価格を求め、消費者が負担する価格上昇分を計算してください。
3. 需要 $Q_d = a - bP$、供給 $Q_s = dP$（$c = 0$）のときの均衡数量を、記号のまま求めてください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
sol1 = sp.solve([sp.Eq(Q, 200 - 4 * P), sp.Eq(Q, 20 + 2 * P)], [P, Q], dict=True)
print(sol1)

# 2
P_tax = sp.solve(sp.Eq(200 - 4 * P, 20 + 2 * (P - 6)), P)[0]
print("課税後の価格:", P_tax, "消費者の負担分:", P_tax - sol1[0][P])

# 3
Q_general = sp.solve([sp.Eq(Q, a - b * P), sp.Eq(Q, d * P)], [P, Q], dict=True)
print(Q_general[0][Q])
```

</details>

---
## 3. 微分：限界概念と弾力性

経済学の「限界〜（marginal）」は、すべて微分です。

| 概念 | 定義 |
|---|---|
| 限界費用 MC | 総費用 TC を数量 Q で微分 |
| 限界収入 MR | 総収入 TR を Q で微分 |
| 限界効用 MU | 効用 U を消費量で微分 |
| 弾力性 | $\dfrac{dQ}{dP}\cdot\dfrac{P}{Q}$ |

### 3.1 sp.diff：限界費用と平均費用

In [ ]:
TC = 100 + 10 * Q + sp.Rational(1, 2) * Q**2     # 総費用関数
MC = sp.diff(TC, Q)                               # 限界費用 = dTC/dQ
AC = TC / Q                                       # 平均費用

print("総費用 TC =", TC)
print("限界費用 MC =", MC)
print("平均費用 AC =", sp.simplify(AC))
print("Q = 20 のとき MC =", MC.subs(Q, 20), ", AC =", AC.subs(Q, 20))

### 3.2 限界収入と限界効用

In [ ]:
# 需要曲線 P = 100 - Q をもつ企業の総収入と限界収入
P_inv = 100 - Q           # 逆需要関数
TR = P_inv * Q            # 総収入
MR = sp.diff(TR, Q)
print("総収入 TR =", sp.expand(TR))
print("限界収入 MR =", MR)

# 限界効用（効用関数 U = ln(x)）：消費量が増えるほど限界効用は減る（限界効用逓減）
U = sp.log(x)
MU = sp.diff(U, x)
print("限界効用 MU =", MU)
print("x=1, 2, 4 のときの MU:", [MU.subs(x, v) for v in (1, 2, 4)])

### 3.3 需要の価格弾力性

需要の価格弾力性は $\varepsilon = \dfrac{dQ}{dP}\cdot\dfrac{P}{Q}$ で定義されます。
（絶対値が 1 より大きければ「弾力的」、小さければ「非弾力的」）

In [ ]:
Qd = 100 - 2 * P
elasticity = sp.diff(Qd, P) * P / Qd
print("弾力性の式:", sp.simplify(elasticity))

for price in (10, 25, 30, 40):
    e = elasticity.subs(P, price)
    label = "弾力的" if abs(e) > 1 else "非弾力的"
    print(f"P = {price}: ε = {e} → {label}")

### 3.4 高階微分とグラフ

`sp.diff(式, 記号, 2)` で 2 階微分が求まります。限界費用が増加している（MC の傾きが正）ことを確認し、
費用曲線をグラフにしてみましょう。

In [ ]:
print("MC の傾き（TC の 2 階微分）:", sp.diff(TC, Q, 2))

q_vals = np.linspace(1, 40, 100)
plt.figure(figsize=(7, 4.5))
plt.plot(q_vals, sp.lambdify(Q, MC)(q_vals), label="限界費用 MC")
plt.plot(q_vals, sp.lambdify(Q, AC)(q_vals), label="平均費用 AC")
plt.xlabel("生産量 Q")
plt.ylabel("費用")
plt.title("限界費用と平均費用")
plt.legend()
plt.grid(True)
plt.show()

### 3.5 微分係数の意味を数値で確かめる

限界費用 $MC(Q)$ は「$Q$ をほんの少し増やしたときの費用の増分」です。
差分 $\dfrac{TC(Q + h) - TC(Q)}{h}$ で $h$ を小さくしていくと、微分の値に近づくことを確かめましょう。

In [ ]:
h = sp.symbols("h", positive=True)
difference_quotient = (TC.subs(Q, Q + h) - TC) / h
print("差分商:", sp.simplify(difference_quotient))
print("h → 0 の極限:", sp.limit(difference_quotient, h, 0))
print("MC(20) =", MC.subs(Q, 20))
for step in (1, sp.Rational(1, 10), sp.Rational(1, 100)):
    print(f"h = {step}: 差分商 = {difference_quotient.subs({Q: 20, h: step})}")

### 練習問題 3

1. 総費用関数 $TC = 50 + 4Q + Q^3/30$ の限界費用と平均費用を求め、$Q = 10$ での値を表示してください。
2. 効用関数 $U = x^{0.5}$（`sp.sqrt(x)`）の限界効用を求め、$x = 1, 4, 9$ での値を表示してください。
3. 需要関数 $Q_d = 300 - 3P$ の価格弾力性が $-1$ になる価格（単位弾力的になる価格）を求めてください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
TC2 = 50 + 4 * Q + Q**3 / 30
MC2 = sp.diff(TC2, Q)
print(MC2, MC2.subs(Q, 10), (TC2 / Q).subs(Q, 10))

# 2
MU2 = sp.diff(sp.sqrt(x), x)
print(MU2, [MU2.subs(x, v) for v in (1, 4, 9)])

# 3
Qd3 = 300 - 3 * P
e3 = sp.diff(Qd3, P) * P / Qd3
print(sp.solve(sp.Eq(e3, -1), P))
```

</details>

---
## 4. 偏微分：生産関数と限界生産力

変数が 2 つ以上ある関数を、1 つの変数だけで微分するのが **偏微分** です。
`sp.diff(式, 記号)` で、指定した記号以外は定数として扱われます。

### 4.1 コブ・ダグラス型生産関数

$F(K, L) = A K^{\alpha} L^{\beta}$ の限界生産力（資本・労働を 1 単位増やしたときの生産量の増分）を求めます。

In [ ]:
A, K, L, alpha, beta = sp.symbols("A K L alpha beta", positive=True)
F = A * K**alpha * L**beta

MPK = sp.diff(F, K)     # 資本の限界生産力
MPL = sp.diff(F, L)     # 労働の限界生産力
display(MPK)
display(MPL)

### 4.2 規模に関する収穫

資本と労働を同時に $t$ 倍したとき、生産量が何倍になるかを調べます。
$F(tK, tL) / F(K, L)$ を簡約すると $t^{\alpha+\beta}$ になり、$\alpha + \beta$ が 1 より大きいか小さいかで
収穫逓増・一定・逓減が決まります。

In [ ]:
t = sp.symbols("t", positive=True)
ratio = sp.simplify(F.subs({K: t * K, L: t * L}) / F)
display(ratio)

# 具体例: alpha = 0.3, beta = 0.7 なら収穫一定
print(ratio.subs({alpha: sp.Rational(3, 10), beta: sp.Rational(7, 10)}))

### 4.3 限界代替率（MRS）

効用関数 $U(x, y)$ について、限界代替率 $MRS = \dfrac{\partial U/\partial x}{\partial U/\partial y}$ は
「$x$ を 1 単位増やす代わりに、$y$ をどれだけ減らしても同じ満足度か」を表します。

In [ ]:
U = x**sp.Rational(1, 2) * y**sp.Rational(1, 2)
MRS = sp.simplify(sp.diff(U, x) / sp.diff(U, y))
display(MRS)

# 2 階の偏微分（ヘッセ行列）：最適化の 2 階条件で使う（第 5 章）
display(sp.hessian(U, (x, y)))

### 練習問題 4

1. 生産関数 $F = 10 K^{0.4} L^{0.6}$（`sp.Rational` を使う）について、$K = 16, L = 81$ のときの資本と労働の限界生産力を求めてください。
2. 効用関数 $U = x^2 y$ の限界代替率を求め、$x = 2, y = 5$ での値を表示してください。
3. 生産関数 $F = K^{0.5} L^{0.6}$ は、規模に関して収穫逓増・一定・逓減のどれか、`ratio` の指数から判断してください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
F1 = 10 * K**sp.Rational(2, 5) * L**sp.Rational(3, 5)
print(sp.diff(F1, K).subs({K: 16, L: 81}), sp.diff(F1, L).subs({K: 16, L: 81}))

# 2
U2 = x**2 * y
MRS2 = sp.simplify(sp.diff(U2, x) / sp.diff(U2, y))
print(MRS2, MRS2.subs({x: 2, y: 5}))

# 3
F3 = K**sp.Rational(1, 2) * L**sp.Rational(3, 5)
print(sp.simplify(F3.subs({K: t * K, L: t * L}) / F3))   # t**(11/10) → 指数が 1 より大きいので収穫逓増
```

</details>

---
## 5. 最適化：一階条件・二階条件・ラグランジュ乗数法

### 5.1 利潤最大化（1 変数）

利潤 $\pi = TR - TC$ を最大にする生産量は、**一階条件** $d\pi/dQ = 0$（つまり $MR = MC$）を解いて求めます。
最大化であることを確かめるには **二階条件** $d^2\pi/dQ^2 < 0$ を確認します。

In [ ]:
TR = (100 - Q) * Q                                # 逆需要 P = 100 - Q
TC = 100 + 10 * Q + sp.Rational(1, 2) * Q**2
profit = TR - TC

foc = sp.diff(profit, Q)                          # 一階条件
Q_opt = sp.solve(sp.Eq(foc, 0), Q)[0]
soc = sp.diff(profit, Q, 2)                       # 二階条件

print("利潤 π =", sp.expand(profit))
print("一階条件 dπ/dQ =", foc)
print("最適生産量 Q* =", Q_opt)
print("二階条件 d²π/dQ² =", soc, "→ 負なので最大")
print("最大利潤 =", profit.subs(Q, Q_opt), ", 価格 =", (100 - Q).subs(Q, Q_opt))

### 5.2 多変数の最適化

2 つの財を生産する企業の利潤最大化では、各変数についての偏微分をすべて 0 とおいた連立方程式を解きます。
二階条件は **ヘッセ行列** が負定値（ここでは対角成分が負で行列式が正）であることで確認します。

In [ ]:
q1, q2 = sp.symbols("q1 q2", positive=True)
profit2 = 60 * q1 + 80 * q2 - (q1**2 + q1 * q2 + 2 * q2**2)

foc2 = [sp.diff(profit2, q1), sp.diff(profit2, q2)]
sol2 = sp.solve(foc2, [q1, q2], dict=True)[0]
print("一階条件:", foc2)
print("最適解:", sol2)

H = sp.hessian(profit2, (q1, q2))
display(H)
print("行列式:", H.det(), "（対角成分が負で行列式が正 → 最大）")
print("最大利潤:", profit2.subs(sol2))

### 5.3 ラグランジュ乗数法：予算制約付き効用最大化

消費者は予算制約 $p_x x + p_y y = I$ のもとで効用 $U(x, y)$ を最大化します。
ラグランジュ関数 $\mathcal{L} = U + \lambda (I - p_x x - p_y y)$ を作り、$x, y, \lambda$ で偏微分して 0 とおきます。
記号のまま解くと **需要関数** が得られます。

In [ ]:
px, py, I, lam = sp.symbols("p_x p_y I lambda", positive=True)
U = x**sp.Rational(1, 2) * y**sp.Rational(1, 2)

lagrangian = U + lam * (I - px * x - py * y)
conditions = [sp.diff(lagrangian, v) for v in (x, y, lam)]
demand = sp.solve(conditions, [x, y, lam], dict=True)[0]

print("x の需要関数:", demand[x])
print("y の需要関数:", demand[y])
print("具体例（p_x=2, p_y=3, I=120）:", {k: v.subs({px: 2, py: 3, I: 120}) for k, v in demand.items()})

コブ・ダグラス型効用関数では、所得の半分ずつを各財に支出する（$x = I / 2p_x$）という有名な結果が、
記号計算で自動的に導かれました。

### 5.4 費用最小化

生産量 $Q_0$ を達成するための費用 $C = wL + rK$ を最小化する問題も、同じ手順で解けます。

In [ ]:
w, r, Q0 = sp.symbols("w r Q_0", positive=True)
cost = w * L + r * K
production = K**sp.Rational(1, 2) * L**sp.Rational(1, 2)

lagrangian_c = cost + lam * (Q0 - production)
sol_c = sp.solve([sp.diff(lagrangian_c, v) for v in (K, L, lam)], [K, L, lam], dict=True)[0]

print("最適な資本 K* =", sol_c[K])
print("最適な労働 L* =", sol_c[L])
print("最小費用 C* =", sp.simplify(cost.subs(sol_c)))

### 練習問題 5

1. 逆需要関数 $P = 120 - 2Q$、総費用 $TC = 20Q + Q^2$ の企業について、利潤を最大にする $Q$ と、そのときの価格・利潤を求めてください。二階条件も確認してください。
2. 効用関数 $U = x^{0.3} y^{0.7}$（`sp.Rational` を使う）、予算制約 $2x + 4y = 100$ のもとでの最適な消費量を、ラグランジュ乗数法で求めてください。
3. 問 2 の解を使って、所得 $I$ が一般の記号のとき $x$ の需要関数がどうなるか（`I` を記号のままにして）求めてください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
# 1
profit1 = (120 - 2 * Q) * Q - (20 * Q + Q**2)
Q1 = sp.solve(sp.diff(profit1, Q), Q)[0]
print(Q1, (120 - 2 * Q).subs(Q, Q1), profit1.subs(Q, Q1), sp.diff(profit1, Q, 2))

# 2
U2 = x**sp.Rational(3, 10) * y**sp.Rational(7, 10)
L2 = U2 + lam * (100 - 2 * x - 4 * y)
print(sp.solve([sp.diff(L2, v) for v in (x, y, lam)], [x, y, lam], dict=True)[0])

# 3
L3 = U2 + lam * (I - 2 * x - 4 * y)
print(sp.solve([sp.diff(L3, v) for v in (x, y, lam)], [x, y, lam], dict=True)[0][x])
```

</details>

---
## 6. 積分：消費者余剰と生産者余剰

### 6.1 sp.integrate：不定積分と定積分

微分の逆が積分です。限界費用を積分すると（固定費用を除いた）総費用に戻ります。
定積分は `sp.integrate(式, (記号, 下限, 上限))` と書きます。

In [ ]:
MC = 10 + Q                       # 限界費用
print("不定積分:", sp.integrate(MC, Q))                 # 可変費用（積分定数 = 固定費用は省略される）
print("Q = 0 から 20 までの定積分:", sp.integrate(MC, (Q, 0, 20)))   # 20 単位生産するときの可変費用

### 6.2 消費者余剰と生産者余剰

需要曲線 $P_d(Q) = 100 - 2Q$、供給曲線 $P_s(Q) = 10 + Q$ の市場では、

- 消費者余剰 CS = $\int_0^{Q^*} (P_d(Q) - P^*)\, dQ$（需要曲線と均衡価格の間の面積）
- 生産者余剰 PS = $\int_0^{Q^*} (P^* - P_s(Q))\, dQ$（均衡価格と供給曲線の間の面積）

In [ ]:
Pd = 100 - 2 * Q      # 逆需要関数
Ps = 10 + Q           # 逆供給関数

Q_star = sp.solve(sp.Eq(Pd, Ps), Q)[0]
P_star = Pd.subs(Q, Q_star)
CS = sp.integrate(Pd - P_star, (Q, 0, Q_star))
PS = sp.integrate(P_star - Ps, (Q, 0, Q_star))

print(f"均衡: Q* = {Q_star}, P* = {P_star}")
print(f"消費者余剰 CS = {CS}, 生産者余剰 PS = {PS}, 総余剰 = {CS + PS}")

In [ ]:
# 余剰をグラフで確認（fill_between で面積を塗る）
q_vals = np.linspace(0, 45, 200)
pd_fn = sp.lambdify(Q, Pd, "numpy")
ps_fn = sp.lambdify(Q, Ps, "numpy")
q_fill = np.linspace(0, float(Q_star), 100)

plt.figure(figsize=(7, 4.5))
plt.plot(q_vals, pd_fn(q_vals), label="需要曲線")
plt.plot(q_vals, ps_fn(q_vals), label="供給曲線")
plt.fill_between(q_fill, pd_fn(q_fill), float(P_star), alpha=0.3, label=f"消費者余剰 = {CS}")
plt.fill_between(q_fill, float(P_star), ps_fn(q_fill), alpha=0.3, label=f"生産者余剰 = {PS}")
plt.axhline(float(P_star), color="gray", linestyle="--")
plt.xlabel("数量 Q")
plt.ylabel("価格 P")
plt.title("消費者余剰と生産者余剰")
plt.legend()
plt.grid(True)
plt.show()

### 練習問題 6

1. 限界費用 $MC = 4 + 0.2Q$ を $Q = 0$ から $50$ まで積分し、50 単位生産するときの可変費用を求めてください。
2. 需要曲線 $P_d = 80 - Q$、供給曲線 $P_s = 20 + 2Q$ の市場の均衡と、消費者余剰・生産者余剰を求めてください。
3. 問 2 の市場で価格が $P = 60$ に規制された（上限価格）とき、取引量は供給量で決まります。そのときの消費者余剰を求めてください（ヒント: 取引量 $Q_c$ は $P_s(Q_c) = 60$ の解、$CS = \int_0^{Q_c}(P_d - 60)\,dQ$）。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
# 1
print(sp.integrate(4 + sp.Rational(1, 5) * Q, (Q, 0, 50)))

# 2
Pd2, Ps2 = 80 - Q, 20 + 2 * Q
Qe = sp.solve(sp.Eq(Pd2, Ps2), Q)[0]
Pe = Pd2.subs(Q, Qe)
print(Qe, Pe, sp.integrate(Pd2 - Pe, (Q, 0, Qe)), sp.integrate(Pe - Ps2, (Q, 0, Qe)))

# 3
Qc = sp.solve(sp.Eq(Ps2, 60), Q)[0]
print(Qc, sp.integrate(Pd2 - 60, (Q, 0, Qc)))
```

</details>

---
## 7. 極限と級数：現在価値と永久年金

### 7.1 sp.limit：連続複利

年利 $r$ を年 $n$ 回に分けて複利計算すると、1 年後の元利合計は $(1 + r/n)^n$ です。
$n \to \infty$ の極限（連続複利）は $e^r$ になります。

In [ ]:
n = sp.symbols("n", positive=True, integer=True)
r = sp.symbols("r", positive=True)

compound = (1 + r / n)**n
print("極限:", sp.limit(compound, n, sp.oo))
print("r = 5% のとき:", [float(compound.subs({r: sp.Rational(5, 100), n: k})) for k in (1, 12, 365)],
      "→ 連続複利:", float(sp.exp(sp.Rational(5, 100))))

### 7.2 sp.summation：年金の現在価値

毎年 $C$ 円を $n$ 年間受け取る年金の現在価値は $PV = \sum_{t=1}^{n} \dfrac{C}{(1+r)^t}$ です。
`sp.summation()` で和を計算し、$n \to \infty$ の極限をとると **永久年金** の現在価値 $C/r$ が得られます。

In [ ]:
C, t_ = sp.symbols("C t", positive=True)

# 具体的な数値: 毎年 100 万円を 10 年間、割引率 5%
pv_10 = sp.summation(100 / (1 + sp.Rational(5, 100))**t_, (t_, 1, 10))
print("10 年間の年金の現在価値（万円）:", round(float(pv_10), 2))

# 記号のまま：n 年間の年金の現在価値
pv_n = sp.summation(C / (1 + r)**t_, (t_, 1, n))
pv_n = sp.simplify(pv_n)
display(pv_n)

# 永久年金（n → ∞）
print("永久年金の現在価値:", sp.simplify(sp.limit(pv_n, n, sp.oo)))

### 7.3 sp.series：テイラー展開

`sp.series()` で関数を多項式で近似できます。たとえば $\ln(1 + x) \approx x$（$x$ が小さいとき）は、
成長率の近似計算（$\ln(Y_t / Y_{t-1}) \approx$ 成長率）でよく使われます。

In [ ]:
print(sp.series(sp.log(1 + x), x, 0, 4))      # 4 次までの展開
print(sp.series(sp.exp(x), x, 0, 4))

# 成長率の近似: 成長率 3% のとき ln(1.03) ≈ 0.03
print("ln(1.03) =", float(sp.log(sp.Rational(103, 100))))

### 7.4 成長率と倍増年数

年率 $g$ で成長する経済が 2 倍になるまでの年数 $n$ は $(1+g)^n = 2$ を解いて $n = \ln 2 / \ln(1+g)$ です。
「70 の法則」（$n \approx 70 / (100g)$）が近似としてよく使われる理由も、$\ln(1+g) \approx g$ と $\ln 2 \approx 0.693$ から説明できます。

In [ ]:
g = sp.symbols("g", positive=True)
n_double = sp.solve(sp.Eq((1 + g)**n, 2), n)[0]
display(n_double)
for rate in (1, 2, 3, 5, 7):
    exact = float(n_double.subs(g, sp.Rational(rate, 100)))
    print(f"成長率 {rate}%: 倍増まで {exact:.1f} 年（70 の法則: {70 / rate:.1f} 年）")

### 練習問題 7

1. 年利 8% を月複利（$n = 12$）で運用したときの 1 年後の元利合計（元本 1）と、連続複利のときの値を比べてください。
2. 毎年 50 万円を 20 年間受け取る年金の現在価値を、割引率 3% で計算してください。
3. 永久年金の現在価値 $C/r$ を使って、毎年 30 万円を永久に受け取れる資産の価値を割引率 2% で求めてください。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```python
# 1
print(float((1 + sp.Rational(8, 100) / 12)**12), float(sp.exp(sp.Rational(8, 100))))

# 2
print(round(float(sp.summation(50 / (1 + sp.Rational(3, 100))**t_, (t_, 1, 20))), 2))

# 3
print(30 / sp.Rational(2, 100))
```

</details>

---
## 8. 行列：連立方程式と投入産出分析

### 8.1 sp.Matrix

`sp.Matrix()` で行列を作り、`.det()`（行列式）、`.inv()`（逆行列）、`.T`（転置）などが使えます。
連立方程式 $A\mathbf{x} = \mathbf{b}$ は $\mathbf{x} = A^{-1}\mathbf{b}$ で解けます。

In [ ]:
A_mat = sp.Matrix([[2, 1], [1, 3]])
b_vec = sp.Matrix([5, 10])

display(A_mat)
print("行列式:", A_mat.det())
display(A_mat.inv())
print("解 x =", (A_mat.inv() * b_vec).T)     # 2x + y = 5, x + 3y = 10 の解
print("LUsolve でも同じ:", A_mat.LUsolve(b_vec).T)

### 8.2 簡単なマクロモデル（IS-LM）を行列で解く

$Y = C + I + G$、$C = 20 + 0.8Y$、$I = 50 - 10 r_i$、$G = 30$、貨幣市場 $0.5Y - 20 r_i = 100$ という体系は、
未知数 $(Y, r_i)$ についての連立 1 次方程式です。`sp.linsolve()` や行列で解けます。

In [ ]:
Y, ri = sp.symbols("Y r_i")
is_curve = sp.Eq(Y, 20 + sp.Rational(8, 10) * Y + 50 - 10 * ri + 30)
lm_curve = sp.Eq(sp.Rational(1, 2) * Y - 20 * ri, 100)

print(sp.solve([is_curve, lm_curve], [Y, ri], dict=True))

### 8.3 投入産出分析（レオンチェフ・モデル）

投入係数行列 $A$（各産業が 1 単位生産するのに他産業の生産物をどれだけ使うか）と最終需要 $\mathbf{d}$ が与えられたとき、
必要な総生産量は $\mathbf{x} = (I - A)^{-1}\mathbf{d}$ で求まります。$(I - A)^{-1}$ を **レオンチェフ逆行列** といいます。

In [ ]:
# 2 産業（農業・工業）の投入係数行列
A_io = sp.Matrix([[sp.Rational(2, 10), sp.Rational(3, 10)],
                  [sp.Rational(4, 10), sp.Rational(1, 10)]])
d_final = sp.Matrix([100, 200])          # 最終需要

leontief_inv = (sp.eye(2) - A_io).inv()
x_total = leontief_inv * d_final

print("レオンチェフ逆行列:")
display(leontief_inv)
print("必要な総生産量（農業, 工業）:", [round(float(v), 1) for v in x_total])

# 最終需要が工業で 10 増えたときの波及効果
print("波及効果:", [round(float(v), 2) for v in leontief_inv * sp.Matrix([0, 10])])

### 練習問題 8

1. 行列 $\begin{pmatrix} 3 & 1 \\ 2 & 4 \end{pmatrix}$ の行列式と逆行列を求めてください。
2. 連立方程式 $3x + y = 9,\ 2x + 4y = 16$ を行列で解いてください。
3. 投入係数行列 $A = \begin{pmatrix} 0.1 & 0.2 \\ 0.3 & 0.2 \end{pmatrix}$、最終需要 $\mathbf{d} = (50, 80)$ のときの総生産量を求めてください。

In [ ]:
# 練習問題 8 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 8 の解答例を見る</strong></summary>

```python
# 1
M = sp.Matrix([[3, 1], [2, 4]])
print(M.det()); display(M.inv())

# 2
print((M.inv() * sp.Matrix([9, 16])).T)

# 3
A3 = sp.Matrix([[sp.Rational(1, 10), sp.Rational(2, 10)], [sp.Rational(3, 10), sp.Rational(2, 10)]])
print([round(float(v), 2) for v in (sp.eye(2) - A3).inv() * sp.Matrix([50, 80])])
```

</details>

---
## 9. 数値化とグラフ：evalf・lambdify・matplotlib

### 9.1 evalf / N / nsolve

SymPy は分数や $\sqrt{2}$ を **正確なまま** 保持します。数値（小数）が欲しいときは `.evalf()` または `sp.N()` を使います。
解析的に解けない方程式は `sp.nsolve()` で数値的に解きます（初期値が必要です）。

In [ ]:
val = sp.sqrt(2) + sp.Rational(1, 3)
print(val, "→", val.evalf(), "→ 小数第 3 位まで:", sp.N(val, 4))

# 解析的に解きにくい方程式: e^x = 3x（初期値 0.5 と 1.5 で 2 つの解を探す）
eq = sp.exp(x) - 3 * x
print(sp.nsolve(eq, x, 0.5), sp.nsolve(eq, x, 1.5))

### 9.2 lambdify：SymPy の式を NumPy 関数に変換する

`sp.lambdify(記号, 式, "numpy")` で、SymPy の式を NumPy 配列を受け取れる高速な関数に変換できます。
グラフを描くときの定番の手順です。

In [ ]:
profit = (100 - Q) * Q - (100 + 10 * Q + sp.Rational(1, 2) * Q**2)
profit_fn = sp.lambdify(Q, profit, "numpy")
Q_opt = sp.solve(sp.diff(profit, Q), Q)[0]

q_vals = np.linspace(0, 60, 200)
plt.figure(figsize=(7, 4.5))
plt.plot(q_vals, profit_fn(q_vals), label="利潤 π(Q)")
plt.axvline(float(Q_opt), color="red", linestyle="--", label=f"最適生産量 Q* = {Q_opt}")
plt.xlabel("生産量 Q")
plt.ylabel("利潤")
plt.title("利潤関数と最適生産量")
plt.legend()
plt.grid(True)
plt.show()

### 練習問題 9

1. $\sqrt{3}$ と $\pi$（`sp.pi`）の和を小数第 6 位まで表示してください。
2. 方程式 $x^3 - 2x - 5 = 0$ の実数解を `sp.nsolve` で求めてください（初期値 2）。
3. 効用関数 $U = \ln(x)$ を `lambdify` で関数化し、$x = 0.5$ から $10$ までのグラフを描いてください。

In [ ]:
# 練習問題 9 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 9 の解答例を見る</strong></summary>

```python
# 1
print(sp.N(sp.sqrt(3) + sp.pi, 7))

# 2
print(sp.nsolve(x**3 - 2 * x - 5, x, 2))

# 3
u_fn = sp.lambdify(x, sp.log(x), "numpy")
xs = np.linspace(0.5, 10, 100)
plt.plot(xs, u_fn(xs))
plt.xlabel("消費量 x"); plt.ylabel("効用 U"); plt.title("U = ln(x)")
plt.grid(True); plt.show()
```

</details>

---
## 10. まとめ

| トピック | 主な関数 | 経済学での使いどころ |
|---|---|---|
| 記号と式 | `sp.symbols()`, `sp.expand()`, `sp.factor()`, `sp.simplify()`, `.subs()` | 関数の定義・整理 |
| 方程式 | `sp.Eq()`, `sp.solve()`, `sp.nsolve()` | 市場均衡、一階条件 |
| 微分 | `sp.diff(式, 記号)`, `sp.diff(式, 記号, 2)` | 限界費用・限界効用・弾力性 |
| 偏微分 | `sp.diff(式, 記号)`, `sp.hessian()` | 限界生産力、MRS、二階条件 |
| 最適化 | 一階条件を `sp.solve()`、ラグランジュ関数 | 利潤最大化、効用最大化、費用最小化 |
| 積分 | `sp.integrate(式, (記号, 下限, 上限))` | 消費者余剰・生産者余剰 |
| 極限・級数 | `sp.limit()`, `sp.summation()`, `sp.series()` | 連続複利、現在価値、近似 |
| 行列 | `sp.Matrix()`, `.inv()`, `.det()`, `sp.eye()` | 連立方程式、投入産出分析 |
| 数値化 | `.evalf()`, `sp.N()`, `sp.lambdify()` | グラフ描画、数値計算 |

## 次のステップ

- `python/scipy/scipy_optimize_beginner_tutorial.ipynb` — 解析的に解けない最適化問題を数値的に解く
- `python/numpy/numpy_beginner_tutorial.ipynb` — 行列計算を NumPy で高速に行う
- `python/statsmodels/statsmodels_tutorial.ipynb` — データから需要関数などを推定する

---
## 総合演習：独占企業の利潤最大化と死荷重

ある財の市場の逆需要関数は $P = 120 - Q$、総費用関数は $TC = 20Q + \dfrac{Q^2}{2}$ です。

1. この市場を **独占企業** 1 社が供給するとき、利潤を最大にする生産量 $Q_m$、価格 $P_m$、利潤を求めてください（一階条件 $MR = MC$）。
2. 同じ費用関数をもつ企業が **完全競争** で供給するとき（$P = MC$）の生産量 $Q_c$ と価格 $P_c$ を求めてください。
3. 独占のときの消費者余剰・生産者余剰と、完全競争のときの総余剰を計算し、**死荷重**（総余剰の差）を求めてください。
4. 需要曲線・限界収入曲線・限界費用曲線を 1 つのグラフに描き、独占の均衡点と完全競争の均衡点を示してください。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください
Q = sp.symbols("Q", positive=True)
P_inv = 120 - Q
TC = 20 * Q + Q**2 / 2

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
Q = sp.symbols("Q", positive=True)
P_inv = 120 - Q
TC = 20 * Q + Q**2 / 2
MR = sp.diff(P_inv * Q, Q)
MC = sp.diff(TC, Q)

# 1. 独占
Q_m = sp.solve(sp.Eq(MR, MC), Q)[0]
P_m = P_inv.subs(Q, Q_m)
profit_m = (P_inv * Q - TC).subs(Q, Q_m)
print(f"独占: Q_m = {Q_m}, P_m = {P_m}, 利潤 = {profit_m}")

# 2. 完全競争
Q_c = sp.solve(sp.Eq(P_inv, MC), Q)[0]
P_c = P_inv.subs(Q, Q_c)
print(f"完全競争: Q_c = {Q_c}, P_c = {P_c}")

# 3. 余剰と死荷重
CS_m = sp.integrate(P_inv - P_m, (Q, 0, Q_m))
PS_m = sp.integrate(P_m - MC, (Q, 0, Q_m))
CS_c = sp.integrate(P_inv - P_c, (Q, 0, Q_c))
PS_c = sp.integrate(P_c - MC, (Q, 0, Q_c))
dwl = (CS_c + PS_c) - (CS_m + PS_m)
print(f"独占: CS = {CS_m}, PS = {PS_m}, 総余剰 = {CS_m + PS_m}")
print(f"完全競争: 総余剰 = {CS_c + PS_c}")
print(f"死荷重 = {dwl}")

# 4. グラフ
q_vals = np.linspace(0, 100, 200)
plt.figure(figsize=(7, 4.5))
plt.plot(q_vals, sp.lambdify(Q, P_inv)(q_vals), label="需要曲線 P = 120 - Q")
plt.plot(q_vals, sp.lambdify(Q, MR)(q_vals), label="限界収入 MR")
plt.plot(q_vals, sp.lambdify(Q, MC)(q_vals), label="限界費用 MC")
plt.scatter([float(Q_m)], [float(P_m)], color="red", zorder=5, label=f"独占 (Q={Q_m}, P={P_m})")
plt.scatter([float(Q_c)], [float(P_c)], color="green", zorder=5, label=f"完全競争 (Q={Q_c}, P={P_c})")
plt.ylim(0, 130)
plt.xlabel("数量 Q")
plt.ylabel("価格 P")
plt.title("独占と完全競争の比較")
plt.legend()
plt.grid(True)
plt.show()

お疲れさまでした！ 経済学の教科書に出てくる計算は、SymPy を使えば「式を書いて `solve` / `diff` / `integrate` を呼ぶ」だけで
再現できます。次は `scipy.optimize` のチュートリアルで、記号では解けない問題を数値的に解く方法を学びましょう。